# 01 Silver Layer: Cleaned Source Tables

This notebook builds the **Silver** layer for the LushProtein project. Silver tables are cleaned and typed versions of the numbered raw-data folders, but they stay close to their source grain for auditability.

Gold analytical marts are created separately in `02_gold_layer.ipynb`. Run this notebook first.


In [15]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "data_cleaning":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "1.customer_transaction").exists():
    for parent in Path.cwd().parents:
        if (parent / "1.customer_transaction").exists():
            PROJECT_ROOT = parent
            break

DATA_DIR = PROJECT_ROOT / "data"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"
for folder in [DATA_DIR, SILVER_DIR, GOLD_DIR]:
    folder.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Silver directory:", SILVER_DIR)
print("Gold directory:", GOLD_DIR)

# Silver owns source-aligned outputs. Remove stale Silver parquet files before rewriting.
for old_file in SILVER_DIR.glob("*.parquet"):
    old_file.unlink()
# Remove legacy root-level parquet files from the previous non-medallion layout.
for old_file in DATA_DIR.glob("*.parquet"):
    old_file.unlink()


Project root: /Users/tohzhengfeng/Documents/SMU/Academia/MITB 2526 Apr/ISSS603 Customer Analytics/Project
Silver directory: /Users/tohzhengfeng/Documents/SMU/Academia/MITB 2526 Apr/ISSS603 Customer Analytics/Project/data/silver
Gold directory: /Users/tohzhengfeng/Documents/SMU/Academia/MITB 2526 Apr/ISSS603 Customer Analytics/Project/data/gold


## 1. Source Context

The glossary and source-to-target mapping are read as documentation. They confirm that Shopify orders require different filters for order-level and line-item-level grains, and that campaign traffic has no customer/order/date key for exact conversion joins.


In [16]:
MAPPING_FILE = PROJECT_ROOT / "LushProtein_Source_to_Target_Mapping_Exercise.xlsx"
GLOSSARY_FILE = PROJECT_ROOT / "LushProtein_Data_Glossary_20260505.xlsx"

for context_file in [MAPPING_FILE, GLOSSARY_FILE]:
    if context_file.exists():
        xls = pd.ExcelFile(context_file)
        print(f"{context_file.name}: {xls.sheet_names}")
    else:
        print(f"Missing context file: {context_file.name}")

if GLOSSARY_FILE.exists():
    glossary_index = pd.read_excel(GLOSSARY_FILE, sheet_name="ReadMe", header=2).dropna(how="all")
    print("\nGlossary index preview:")
    print(glossary_index[["folder", "glossary_sheet", "primary_key"]].to_string(index=False))


LushProtein_Source_to_Target_Mapping_Exercise.xlsx: ['1.orders', '2_1.products_master']
LushProtein_Data_Glossary_20260505.xlsx: ['ReadMe', '1.orders', '2_1.products_master', '3_1.discounts_export', '4_1.Sessions_by_referrer', '5_1.orders_combined', '5_2.order_items_checkout', '5_3.subscribers_reactivated', '5_4.subscriptions_churned', '5_5.order_items_recurring']

Glossary index preview:
                folder              glossary_sheet                                                            primary_key
1.customer_transaction                    1.orders                                      ID (order PK), Line: ID (line PK)
      2.product_master         2_1.products_master                                Handle (product), Variant SKU (variant)
           3.Discounts        3_1.discounts_export                                        Name (the discount code itself)
           4.Campaigns    4_1.Sessions_by_referrer (none — every row is unique by combination of all 8 dimension columns

## 2. Locate Raw Files

Only numbered raw-data folders are used. Temporary Excel lock files and system files are ignored.


In [17]:
def list_source_files(folder, pattern):
    folder_path = PROJECT_ROOT / folder
    files = sorted(
        p for p in folder_path.glob(pattern)
        if p.is_file() and not p.name.startswith("~$") and p.name != ".DS_Store"
    )
    if not files:
        raise FileNotFoundError(f"No files found for {folder}/{pattern}")
    return files

ORDER_FILES = list_source_files("1.customer_transaction", "1_*.xlsx")
PRODUCT_FILE = list_source_files("2.product_master", "*.xlsx")[0]
DISCOUNTS_FILE = list_source_files("3.Discounts", "*.csv")[0]
CAMPAIGNS_FILE = list_source_files("4.Campaigns", "*.csv")[0]
RECHARGE_FILES = {
    "recharge_orders": PROJECT_ROOT / "5.Recharge_data" / "5_1.orders_combined_20260505.xlsx",
    "recharge_checkout_items": PROJECT_ROOT / "5.Recharge_data" / "5_2.order_items_checkout_20260505.xlsx",
    "recharge_reactivated": PROJECT_ROOT / "5.Recharge_data" / "5_3.subscribers_reactivated_20260505.xlsx",
    "recharge_churned": PROJECT_ROOT / "5.Recharge_data" / "5_4.subscriptions_churned_20260505.xlsx",
    "recharge_recurring_items": PROJECT_ROOT / "5.Recharge_data" / "5_5.order_items_recurring_20260505.xlsx",
}

print("Shopify order files:", len(ORDER_FILES))
for path in ORDER_FILES:
    print(" -", path.relative_to(PROJECT_ROOT))
print("Product master:", PRODUCT_FILE.relative_to(PROJECT_ROOT))
print("Discounts:", DISCOUNTS_FILE.relative_to(PROJECT_ROOT))
print("Campaigns:", CAMPAIGNS_FILE.relative_to(PROJECT_ROOT))
for name, path in RECHARGE_FILES.items():
    if not path.exists():
        raise FileNotFoundError(path)
    print(f"{name}:", path.relative_to(PROJECT_ROOT))


Shopify order files: 7
 - 1.customer_transaction/1_1.orders-2020_20260505.xlsx
 - 1.customer_transaction/1_2.orders-2021_20260505.xlsx
 - 1.customer_transaction/1_3.orders-2022_20260505.xlsx
 - 1.customer_transaction/1_4.orders-2023_20260505.xlsx
 - 1.customer_transaction/1_5.orders-2024_20260505.xlsx
 - 1.customer_transaction/1_6.orders-2025_20260505.xlsx
 - 1.customer_transaction/1_7.orders-2026_20260505.xlsx
Product master: 2.product_master/2_1.products_master_20260505.xlsx
Discounts: 3.Discounts/3_1.discounts_export_20260505.csv
Campaigns: 4.Campaigns/4_1.Sessions by referrer_20260505.csv
recharge_orders: 5.Recharge_data/5_1.orders_combined_20260505.xlsx
recharge_checkout_items: 5.Recharge_data/5_2.order_items_checkout_20260505.xlsx
recharge_reactivated: 5.Recharge_data/5_3.subscribers_reactivated_20260505.xlsx
recharge_churned: 5.Recharge_data/5_4.subscriptions_churned_20260505.xlsx
recharge_recurring_items: 5.Recharge_data/5_5.order_items_recurring_20260505.xlsx


## 3. Shared Cleaning Helpers

IDs are kept as strings, dates are parsed, and mixed object columns are standardized before parquet writes. SG/MY/HK money fields are normalized to SGD for cross-market analysis.


In [18]:
FX_RATES_TO_SGD = {
    "SG": 1.0,
    "MY": 1.0 / 3.30,  # 1 SGD = 3.30 MYR
    "HK": 1.0 / 6.10,  # 1 SGD = 6.10 HKD
}
ANALYSIS_DATE = pd.Timestamp("2026-04-30", tz="Asia/Singapore")

PRODUCT_MAP = {
    "lean-protein": "Lean Protein",
    "lean_protein": "Lean Protein",
    "clear-protein": "Clear Protein",
    "clear_protein": "Clear Protein",
    "collagen": "Collagen Glow",
    "soy-protein": "Soy Protein",
    "protein-bar": "Protein Bar",
    "shaker": "Accessories",
    "starter-kit": "Accessories",
    "multivitamin": "Supplements",
    "cap-": "Supplements",
}
MARKETPLACE_KEYWORDS = ["shopee", "lazada", "tokopedia", "redmart", "grab"]


def clean_id(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if text == "" or text.lower() in {"nan", "none", "nat"}:
        return pd.NA
    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]
    return text


def is_array_value(value):
    return isinstance(value, (list, tuple, np.ndarray))


def clean_object_cols(df):
    df = df.copy()
    for col in df.select_dtypes(include=["object"]).columns:
        non_null = df[col].dropna()
        if not non_null.empty and non_null.map(is_array_value).any():
            # Preserve list-like aggregation columns as parquet arrays instead of converting them to strings.
            df[col] = df[col].map(lambda x: list(x) if is_array_value(x) else ([] if pd.isna(x) or str(x).strip() == "" else [str(x).strip()]))
        else:
            df[col] = df[col].map(lambda x: pd.NA if pd.isna(x) or str(x).strip() == "" else str(x).strip())
    return df


def parse_numeric_cols(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def store_prefix(name):
    if pd.isna(name):
        return "Unknown"
    n = str(name).upper().replace("#", "").strip()
    if n.startswith("LPMY"):
        return "MY"
    if n.startswith("LPHK"):
        return "HK"
    if n.startswith("LPSG"):
        return "SG"
    if n.startswith("LP"):
        return "SG"
    return "Other"


def classify_product(handle):
    if pd.isna(handle):
        return "Unknown"
    h = str(handle).lower()
    for keyword, label in PRODUCT_MAP.items():
        if keyword in h:
            return label
    return "Other"


def classify_channel(row):
    tags = str(row.get("Tags", "") or "").lower()
    utm = str(row.get("Browser: UTM Source", "") or "").lower()
    name = str(row.get("Name", "") or "").lower()

    if any(keyword in tags for keyword in MARKETPLACE_KEYWORDS):
        return "Marketplace"
    if "subscription" in tags or "yotpo subscriptions" in tags or name.startswith("lpsg"):
        return "Subscription"
    if utm in {"facebook", "instagram", "tiktok"}:
        return "Paid Social"
    if utm in {"google", "bing"}:
        return "Paid Search"
    if utm == "affiliate":
        return "Affiliate"
    if utm in {"shopify_email", "email", "klaviyo"}:
        return "Email"
    return "Direct / Organic"


def save_parquet(df, folder, filename):
    path = folder / filename
    clean_df = clean_object_cols(df)
    clean_df.to_parquet(path, index=False)
    reloaded = pd.read_parquet(path)
    print(f"Saved {path.parent.name}/{filename}: {len(reloaded):,} rows, {len(reloaded.columns):,} columns")
    return path



def validation_summary(name, df, key_cols=None, expected_grain=None, required_cols=None, money_cols=None):
    print(f"\n[{name}] {len(df):,} rows x {len(df.columns):,} columns")
    if expected_grain:
        print(f" - expected grain: {expected_grain}")
    if required_cols:
        missing = [col for col in required_cols if col not in df.columns]
        print(f" - required columns present: {not missing}")
        if missing:
            raise AssertionError(f"{name} missing required columns: {missing}")
    if key_cols:
        missing_keys = [col for col in key_cols if col not in df.columns]
        if missing_keys:
            raise AssertionError(f"{name} missing key columns: {missing_keys}")
        key_nulls = df[key_cols].isna().any(axis=1).sum()
        duplicate_keys = df.duplicated(subset=key_cols).sum()
        print(f" - key columns: {key_cols}")
        print(f" - rows with null key: {key_nulls:,} ({key_nulls / max(len(df), 1):.1%})")
        print(f" - duplicate key rows: {duplicate_keys:,}")
    if money_cols:
        for col in [c for c in money_cols if c in df.columns]:
            non_numeric = pd.to_numeric(df[col], errors="coerce").isna() & df[col].notna()
            print(f" - money column {col}: non-numeric values {non_numeric.sum():,}")


## 4. Clean Shopify Customer Transactions

The Shopify export mixes several row types under the same order ID. In some Matrixify/Shopify exports, child rows may leave order-level columns blank and implicitly belong to the most recent filled order row. To make the pipeline robust to that layout, this notebook first forward-fills **order-level context within each source workbook** before splitting rows.

Silver separates these grains:

- `orders`: one row per order from `Top Row == 1`.
- `lines`: one row per purchased product line from `Line: Type == "Line Item"`.
- `order_non_product_lines`: shipping, discount, transaction, refund, and fulfilment rows retained for audit.

Revenue rule: customer/order revenue must exclude shipping. The notebook keeps the raw Shopify total fields, but creates explicit SGD columns such as `order_revenue_sgd` and `shipping_revenue_sgd`. Downstream Gold marts use `order_revenue_sgd`, not shipping-inclusive totals.

Fulfilment rows are **not** treated as product demand because they mostly mirror product lines with negative quantities and no useful delivery timing/carrier detail. Shipping geography, shipping fee, and fulfilment status are retained on the order-level table.

In [19]:
ORDER_CONTEXT_COLUMNS = [
    "ID", "Name", "Tags", "Cancelled At", "Cancel: Reason", "Processed At", "Currency", "Source", "Checkout ID",
    "Weight Total", "Price: Total Line Items", "Price: Current Subtotal", "Price: Subtotal", "Price: Total Discount",
    "Price: Current Total Shipping", "Price: Total Shipping", "Price: Total Refund", "Price: Current Total", "Price: Total",
    "Payment: Status", "Order Fulfillment Status", "Purchase Order Number", "Customer: ID", "Customer: Tags",
    "Customer: Email Marketing Status", "Customer: SMS Marketing Status", "Shipping: Zip", "Shipping: City", "Shipping: Province",
    "Shipping: Province Code", "Shipping: Country", "Shipping: Country Code", "Browser: User Agent", "Browser: Landing Page",
    "Browser: Referrer", "Browser: Referrer Domain", "Browser: Ad URL", "Browser: UTM Source", "Browser: UTM Medium",
    "Browser: UTM Campaign", "Browser: UTM Content",
]


def fill_order_context(df):
    """Propagate order header fields down child rows within one source workbook.

    Matrixify-style Shopify exports can represent an order as a block: the first row
    has the order header, and following child rows may omit those header values until
    the next order starts. We construct an order block from the most recent nonblank
    `ID`, then forward-fill order-level fields only within that block. This avoids
    accidentally carrying a previous customer's fields into a new order whose header
    has its own ID but a legitimately missing value.
    """
    df = df.copy()
    raw_id = df["ID"].where(df["ID"].notna() & df["ID"].astype(str).str.strip().ne(""), pd.NA)
    df["order_block_id"] = raw_id.ffill()
    for col in [c for c in ORDER_CONTEXT_COLUMNS if c in df.columns]:
        df[col] = df[col].where(df[col].notna() & df[col].astype(str).str.strip().ne(""), pd.NA)
        df[col] = df.groupby("order_block_id", dropna=False)[col].ffill()
    return df

print("Loading Shopify order workbooks...")
raw_chunks = []
for path in ORDER_FILES:
    df = pd.read_excel(path, dtype={"ID": "string", "Customer: ID": "string", "Line: ID": "string"})
    df["source_file"] = path.name
    df["source_row_number"] = np.arange(2, len(df) + 2)
    missing_id_before = df["ID"].isna().sum() if "ID" in df.columns else 0
    df = fill_order_context(df)
    missing_id_after = df["ID"].isna().sum() if "ID" in df.columns else 0
    raw_chunks.append(df)
    print(f" - {path.name}: {len(df):,} rows | blank ID before fill: {missing_id_before:,}, after fill: {missing_id_after:,}")

raw_orders = pd.concat(raw_chunks, ignore_index=True)
print(f"Total raw Shopify rows: {len(raw_orders):,}")

raw_orders["ID"] = raw_orders["ID"].map(clean_id)
if "Customer: ID" in raw_orders.columns:
    raw_orders["Customer: ID"] = raw_orders["Customer: ID"].map(clean_id)
raw_orders["processed_at_utc"] = pd.to_datetime(raw_orders["Processed At"], errors="coerce", utc=True)
raw_orders["processed_at_sgt"] = raw_orders["processed_at_utc"].dt.tz_convert("Asia/Singapore")
raw_orders["order_date"] = raw_orders["processed_at_sgt"].dt.normalize()
raw_orders["store"] = raw_orders["Name"].map(store_prefix)

print("Raw store split:")
print(raw_orders["store"].value_counts(dropna=False).to_string())

Loading Shopify order workbooks...
 - 1_1.orders-2020_20260505.xlsx: 12,201 rows | blank ID before fill: 0, after fill: 0
 - 1_2.orders-2021_20260505.xlsx: 33,196 rows | blank ID before fill: 0, after fill: 0
 - 1_3.orders-2022_20260505.xlsx: 18,563 rows | blank ID before fill: 0, after fill: 0
 - 1_4.orders-2023_20260505.xlsx: 12,962 rows | blank ID before fill: 0, after fill: 0
 - 1_5.orders-2024_20260505.xlsx: 26,373 rows | blank ID before fill: 0, after fill: 0
 - 1_6.orders-2025_20260505.xlsx: 40,932 rows | blank ID before fill: 0, after fill: 0
 - 1_7.orders-2026_20260505.xlsx: 9,601 rows | blank ID before fill: 0, after fill: 0
Total raw Shopify rows: 153,828
Raw store split:
store
SG       93743
MY       59685
HK         395
Other        5


In [20]:
order_columns = [
    "ID", "Name", "Tags", "order_date", "processed_at_sgt", "store", "Customer: ID", "Currency", "original_currency",
    "Price: Total Line Items", "Price: Current Subtotal", "Price: Subtotal", "Price: Total Discount",
    "Price: Total Shipping", "Price: Current Total Shipping", "Price: Current Total", "Price: Total",
    "Payment: Status", "Order Fulfillment Status", "Shipping: Country", "Shipping: Country Code",
    "Browser: UTM Source", "Browser: UTM Medium", "Browser: UTM Campaign", "Browser: Referrer Domain",
    "Cancelled At", "Cancel: Reason", "Line: Product Handle", "Line: Title", "Line: Variant Title",
    "Line: SKU", "Line: Price", "Line: Quantity", "source_file", "source_row_number",
]
existing_order_columns = [col for col in order_columns if col in raw_orders.columns]
top_row_mask = raw_orders["Top Row"].fillna("").astype(str).str.lower().isin(["1", "1.0", "true"])
orders = raw_orders.loc[top_row_mask, existing_order_columns].copy()
orders = orders.rename(columns={"ID": "order_id", "Customer: ID": "customer_id"})
orders["order_id"] = orders["order_id"].map(clean_id)
orders["customer_id"] = orders["customer_id"].map(clean_id)

before_filter = len(orders)
orders = orders.dropna(subset=["order_id", "customer_id", "order_date"])
if "Payment: Status" in orders.columns:
    orders = orders[orders["Payment: Status"].isin(["paid", "partially_refunded"]) | orders["Payment: Status"].isna()]
if "Order Fulfillment Status" in orders.columns:
    orders = orders[orders["Order Fulfillment Status"].fillna("") != "restocked"]
print(f"Order rows before quality filters: {before_filter:,}")
print(f"Order rows after quality filters:  {len(orders):,}")

money_cols = [
    "Price: Total Line Items", "Price: Current Subtotal", "Price: Subtotal", "Price: Total Discount",
    "Price: Total Shipping", "Price: Current Total Shipping", "Price: Current Total", "Price: Total", "Line: Price",
]
orders = parse_numeric_cols(orders, money_cols + ["Line: Quantity"])
orders["fx_to_sgd"] = orders["store"].map(FX_RATES_TO_SGD).fillna(1.0)
for col in money_cols:
    if col in orders.columns:
        orders[col] = orders[col].fillna(0) * orders["fx_to_sgd"]
orders["original_currency"] = orders.get("Currency", pd.Series(pd.NA, index=orders.index))
orders["Currency"] = "SGD"

orders["shipping_revenue_sgd"] = orders.get("Price: Total Shipping", pd.Series(0, index=orders.index)).fillna(0)
orders["order_total_incl_shipping_sgd"] = orders.get("Price: Total", pd.Series(0, index=orders.index)).fillna(0)
# Customer analytics revenue should exclude shipping. Use the final Shopify order total minus shipping,
# rather than subtotal, so discounts/refunds already reflected in the final total are respected.
orders["order_total_incl_shipping_sgd"] = orders["order_total_incl_shipping_sgd"].fillna(0)
orders["shipping_revenue_sgd"] = orders["shipping_revenue_sgd"].fillna(0)
orders["order_revenue_sgd"] = orders["order_total_incl_shipping_sgd"] - orders["shipping_revenue_sgd"]
orders["order_discount_sgd"] = orders.get("Price: Total Discount", pd.Series(0, index=orders.index)).fillna(0)
orders["order_line_items_gross_sgd"] = orders.get("Price: Total Line Items", pd.Series(np.nan, index=orders.index))
orders = orders.drop(columns=["fx_to_sgd"])

orders["channel"] = orders.apply(classify_channel, axis=1)
orders["product_category"] = orders["Line: Product Handle"].map(classify_product)
orders["has_discount"] = orders["order_discount_sgd"].fillna(0) > 0
orders["is_subscription"] = orders["Tags"].fillna("").str.lower().str.contains("subscription|yotpo subscriptions", regex=True)
orders["is_first_order_tag"] = orders["Tags"].fillna("").str.upper().str.contains("FIRST_ORDER", regex=False)

print("Orders grain checks:")
print(" - rows:", f"{len(orders):,}")
print(" - unique order_id:", f"{orders['order_id'].nunique():,}")
print(" - duplicate order_id rows:", f"{orders['order_id'].duplicated().sum():,}")
print(" - date range:", orders["order_date"].min(), "to", orders["order_date"].max())
print(" - revenue excludes shipping:", bool((orders["order_revenue_sgd"] <= orders["order_total_incl_shipping_sgd"] + 0.01).all()))
print(" - store split:")
print(orders["store"].value_counts(dropna=False).to_string())

Order rows before quality filters: 28,054
Order rows after quality filters:  27,350
Orders grain checks:
 - rows: 27,350
 - unique order_id: 27,350
 - duplicate order_id rows: 0
 - date range: 2020-01-01 00:00:00+08:00 to 2026-03-31 00:00:00+08:00
 - revenue excludes shipping: True
 - store split:
store
SG    16039
MY    11309
HK        2


In [21]:
line_columns = [
    "ID", "Customer: ID", "order_date", "processed_at_sgt", "store", "Currency",
    "Payment: Status", "Order Fulfillment Status", "Shipping: Zip", "Shipping: City", "Shipping: Province",
    "Shipping: Country", "Shipping: Country Code", "Line: ID", "Line: Type", "Line: Product ID",
    "Line: Product Handle", "Line: Title", "Line: Name", "Line: Variant ID", "Line: Variant Title",
    "Line: SKU", "Line: Quantity", "Line: Price", "Line: Discount", "Line: Discount Allocation",
    "Line: Discount per Item", "Line: Total", "Line: Grams", "Line: Requires Shipping", "Line: Vendor",
    "Line: Gift Card", "Line: Variant SKU", "Line: Variant Barcode", "Line: Variant Weight",
    "Line: Variant Weight Unit", "Line: Variant Inventory Qty", "Line: Variant Cost", "Line: Variant Price",
    "source_file", "source_row_number",
]
existing_line_columns = [col for col in line_columns if col in raw_orders.columns]

def clean_shopify_line_table(df):
    df = df.rename(columns={"ID": "order_id", "Customer: ID": "customer_id", "Line: ID": "line_id"})
    for col in ["order_id", "customer_id", "line_id", "Line: Product ID", "Line: Variant ID"]:
        if col in df.columns:
            df[col] = df[col].map(clean_id)
    df = df.dropna(subset=["order_id", "order_date"])
    df = parse_numeric_cols(df, [
        "Line: Quantity", "Line: Price", "Line: Discount", "Line: Discount Allocation", "Line: Discount per Item",
        "Line: Total", "Line: Grams", "Line: Variant Weight", "Line: Variant Inventory Qty", "Line: Variant Cost", "Line: Variant Price",
    ])
    df["fx_to_sgd"] = df["store"].map(FX_RATES_TO_SGD).fillna(1.0)
    for col in ["Line: Price", "Line: Discount", "Line: Discount Allocation", "Line: Discount per Item", "Line: Total", "Line: Variant Cost", "Line: Variant Price"]:
        if col in df.columns:
            df[col] = df[col].fillna(0) * df["fx_to_sgd"]
    df["Currency"] = "SGD"
    return df.drop(columns=["fx_to_sgd"])

lines = raw_orders.loc[raw_orders["Line: Type"] == "Line Item", existing_line_columns].copy()
before_line_filter = len(lines)
lines = clean_shopify_line_table(lines).dropna(subset=["customer_id"])
lines["product_category"] = lines["Line: Product Handle"].map(classify_product)

non_product_lines = raw_orders.loc[raw_orders["Line: Type"].notna() & (raw_orders["Line: Type"] != "Line Item"), existing_line_columns].copy()
non_product_lines = clean_shopify_line_table(non_product_lines)

line_item_summary = (
    lines.assign(
        line_item_label=lambda d: (
            d["Line: Quantity"].fillna(0).astype(float).round(2).astype(str).str.replace(r"\.0$", "", regex=True)
            + " x " + d["Line: Title"].fillna("Unknown")
            + d["Line: Variant Title"].fillna("").map(lambda x: f" ({x})" if x else "")
            + d["Line: SKU"].fillna("").map(lambda x: f" [{x}]" if x else "")
        )
    )
    .sort_values(["order_id", "source_row_number"])
    .groupby("order_id")
    .agg(
        line_item_count=("line_id", "count"),
        line_item_quantity=("Line: Quantity", "sum"),
        line_items=("line_item_label", lambda s: [str(x) for x in s.dropna()]),
        line_item_skus=("Line: SKU", lambda s: sorted({str(x) for x in s.dropna() if str(x).strip()})),
    )
    .reset_index()
)
orders = orders.merge(line_item_summary, on="order_id", how="left", validate="one_to_one")
orders["line_item_count"] = orders["line_item_count"].fillna(0).astype(int)
orders["line_item_quantity"] = orders["line_item_quantity"].fillna(0)
orders["line_items"] = orders["line_items"].map(lambda x: x if isinstance(x, list) else [])
orders["line_item_skus"] = orders["line_item_skus"].map(lambda x: x if isinstance(x, list) else [])

print(f"Line rows before key filters: {before_line_filter:,}")
print(f"Line rows after key filters:  {len(lines):,}")
print("Line items with order match:", f"{lines['order_id'].isin(orders['order_id']).mean():.1%}")
print("Orders with line item summary:", f"{(orders['line_item_count'] > 0).mean():.1%}")
print("\nNon-product Shopify rows preserved in Silver:")
print(non_product_lines["Line: Type"].value_counts(dropna=False).to_string())

Line rows before key filters: 51,220
Line rows after key filters:  50,963
Line items with order match: 98.4%
Orders with line item summary: 100.0%

Non-product Shopify rows preserved in Silver:
Line: Type
Fulfillment Line    50089
Shipping Line       24980
Transaction         17262
Discount             8774
Refund Line          1329
Refund Shipping       174


## 5. Clean Product Master

The product master is preserved as a variant/SKU lookup table. Image-only or description continuation rows without a SKU are removed from the Silver variant table.


In [22]:
products_raw = pd.read_excel(PRODUCT_FILE, dtype="string")
products_raw = clean_object_cols(products_raw)
products = products_raw.copy().rename(columns={
    "Handle": "product_handle",
    "Title": "product_title",
    "Vendor": "vendor",
    "Variant SKU": "product_variant_sku",
    "Variant Price": "variant_price",
    "Cost per item": "cost_per_item",
    "Status": "status",
})
products["product_handle"] = products["product_handle"].map(clean_id)
products["product_variant_sku"] = products["product_variant_sku"].map(clean_id)
products = parse_numeric_cols(products, ["variant_price", "cost_per_item", "Variant Grams", "Price / Singapore", "Price / Hong Kong", "Price / Malaysia"])
products["product_category"] = products["product_handle"].map(classify_product)
products = products.dropna(subset=["product_variant_sku"]).drop_duplicates(subset=["product_variant_sku"], keep="first").copy()
print("Product master variant rows:", f"{len(products):,}")


Product master variant rows: 56


## 6. Clean Discounts And Campaigns

Discounts and campaigns are kept as standalone Silver tables. Campaigns have no order/customer/date key, so they are not joined to transactions.


In [23]:
discounts = pd.read_csv(DISCOUNTS_FILE, encoding="utf-8", encoding_errors="replace")
discounts = clean_object_cols(discounts)
discounts = parse_numeric_cols(discounts, ["Value", "Times Used In Total", "Usage Limit Per Code", "Minimum Purchase Requirements"])
for col in ["Start", "End"]:
    if col in discounts.columns:
        discounts[col] = pd.to_datetime(discounts[col], errors="coerce", utc=True)

campaigns = pd.read_csv(CAMPAIGNS_FILE, encoding="utf-8", encoding_errors="replace")
campaigns = clean_object_cols(campaigns)
campaigns = parse_numeric_cols(campaigns, ["Online store visitors", "Sessions"])

print("Discount rows:", f"{len(discounts):,}")
print("Campaign rows:", f"{len(campaigns):,}")


Discount rows: 367
Campaign rows: 137,033


## 7. Clean Recharge Tables

Recharge files stay separate in Silver because they represent different grains: orders, checkout items, recurring items, reactivations, and churned subscriptions.


In [24]:
def clean_recharge_table(path):
    df = pd.read_excel(path, dtype="string")
    df = clean_object_cols(df)
    for id_col in ["recharge_order_id", "shopify_order_id", "customer_id", "subscription_id", "product_id", "variant_id", "order_id"]:
        if id_col in df.columns:
            df[id_col] = df[id_col].map(clean_id)
    for col in [col for col in df.columns if "date" in col.lower()]:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    df = parse_numeric_cols(df, [
        "order_total", "order_gross_revenue", "order_tax", "order_shipping", "order_discounts",
        "order_item_quantity", "line_item_price", "line_item_tax", "line_item_discount",
    ])
    return df

recharge_orders = clean_recharge_table(RECHARGE_FILES["recharge_orders"])
recharge_checkout_items = clean_recharge_table(RECHARGE_FILES["recharge_checkout_items"])
recharge_recurring_items = clean_recharge_table(RECHARGE_FILES["recharge_recurring_items"])
recharge_reactivated = clean_recharge_table(RECHARGE_FILES["recharge_reactivated"])
recharge_churned = clean_recharge_table(RECHARGE_FILES["recharge_churned"])

for name, df in {
    "recharge_orders": recharge_orders,
    "recharge_checkout_items": recharge_checkout_items,
    "recharge_recurring_items": recharge_recurring_items,
    "recharge_reactivated": recharge_reactivated,
    "recharge_churned": recharge_churned,
}.items():
    print(f"{name}: {len(df):,} rows, {len(df.columns):,} columns")


recharge_orders: 1,215 rows, 10 columns
recharge_checkout_items: 1,094 rows, 14 columns
recharge_recurring_items: 650 rows, 14 columns
recharge_reactivated: 50 rows, 4 columns
recharge_churned: 526 rows, 12 columns


## 8. Save Silver Tables And README


In [25]:
silver_outputs = {
    "orders.parquet": orders,
    "lines.parquet": lines,
    "order_non_product_lines.parquet": non_product_lines,
    "products.parquet": products,
    "discounts.parquet": discounts,
    "campaigns.parquet": campaigns,
    "recharge_orders.parquet": recharge_orders,
    "recharge_checkout_items.parquet": recharge_checkout_items,
    "recharge_recurring_items.parquet": recharge_recurring_items,
    "recharge_reactivated.parquet": recharge_reactivated,
    "recharge_churned.parquet": recharge_churned,
}

saved_paths = [save_parquet(df, SILVER_DIR, filename) for filename, df in silver_outputs.items()]

silver_readme = """# Silver Layer

Silver contains cleaned, source-aligned tables. These tables have parsed dates, cleaned IDs, consistent object columns, and SGD-normalized money fields where relevant. They are still close to the original source grains and are intended for auditability and reusable downstream modeling.

| Dataset | Grain | What it contains | Notes |
|---|---:|---|---|
| `orders.parquet` | 1 row per Shopify order | Cleaned order headers, customer ID, order date, store, channel, payment/fulfilment status, shipping geography, shipping fee, non-shipping order revenue, discount totals, subscription flag, and a array-based `line_items` and `line_item_skus` summary columns | Use for customer value, cohorts, retention, geography, channel, and shipping-fee analysis. Use `order_revenue_sgd` for revenue because it excludes shipping. |
| `lines.parquet` | 1 row per Shopify product line item | Product line items from `Line: Type == Line Item`, including SKU, handle, quantity, line value excluding shipping, and product category | Use for product/category/basket analysis. |
| `order_non_product_lines.parquet` | 1 row per non-product Shopify row | Shipping lines, discount rows, transaction rows, refund rows, and fulfilment lines | Preserved for audit. Do not mix fulfilment lines into product demand metrics because they mirror product lines with negative quantities. |
| `products.parquet` | 1 row per product variant SKU | Cleaned product master variant lookup | Used to enrich line items by SKU when a reliable SKU match exists. |
| `discounts.parquet` | 1 row per discount code | Discount metadata, value/type/status/usage | Kept separate unless an order-level discount-code key exists. |
| `campaigns.parquet` | 1 row per aggregate referrer/session combination | Pre-aggregated traffic by referrer, UTM, landing page, city, visitors, sessions | No date/customer/order key, so it is not joined to orders. |
| `recharge_orders.parquet` | 1 row per Recharge order | Subscription order totals and Shopify order IDs | Kept at Recharge order grain. |
| `recharge_checkout_items.parquet` | 1 row per Recharge checkout item | First subscription checkout item lines | Kept separate from recurring items. |
| `recharge_recurring_items.parquet` | 1 row per Recharge recurring item | Subscription renewal item lines | Kept separate from checkout items. |
| `recharge_reactivated.parquet` | 1 row per reactivation event | Subscriber reactivation dates | Recharge customer IDs are not assumed to equal Shopify customer IDs. |
| `recharge_churned.parquet` | 1 row per churned subscription record | Subscription churn dates, SKU/title, churn type, cancellation reason | Subscription grain; not merged to other Recharge files by default. |

## Customer Transaction Grain Decision

For customer analytics, order headers and product line items are necessary. Shipping geography, shipping fee, and fulfilment status are retained on `orders.parquet`; revenue analysis should use `order_revenue_sgd`, which excludes shipping. Detailed `Shipping Line`, `Discount`, `Refund Line`, and `Refund Shipping` rows can support operational diagnostics, so they are preserved in `order_non_product_lines.parquet`. `Fulfillment Line` rows are not promoted to Gold because they do not contain delivery timing/carrier detail in this export and would double-count or reverse product quantities if mixed with product line items.
"""
(SILVER_DIR / "README.md").write_text(silver_readme, encoding="utf-8")
print("Saved README:", (SILVER_DIR / "README.md").relative_to(PROJECT_ROOT))


Saved silver/orders.parquet: 27,350 rows, 49 columns
Saved silver/lines.parquet: 50,963 rows, 42 columns
Saved silver/order_non_product_lines.parquet: 102,608 rows, 41 columns
Saved silver/products.parquet: 56 rows, 25 columns
Saved silver/discounts.parquet: 367 rows, 17 columns
Saved silver/campaigns.parquet: 137,033 rows, 10 columns
Saved silver/recharge_orders.parquet: 1,215 rows, 10 columns
Saved silver/recharge_checkout_items.parquet: 1,094 rows, 14 columns
Saved silver/recharge_recurring_items.parquet: 650 rows, 14 columns
Saved silver/recharge_reactivated.parquet: 50 rows, 4 columns
Saved silver/recharge_churned.parquet: 526 rows, 12 columns
Saved README: data/silver/README.md


## 9. Silver Validation

These checks verify every Silver dataset: expected grain, key coverage, duplicate-key reporting, required columns, numeric money/count fields, parquet read-back, and the non-shipping revenue rule.

In [26]:
print("Silver validation checks")

validation_summary(
    "orders",
    orders,
    key_cols=["order_id"],
    expected_grain="one row per cleaned Shopify order",
    required_cols=["order_id", "customer_id", "order_date", "order_revenue_sgd", "shipping_revenue_sgd", "order_total_incl_shipping_sgd", "line_items"],
    money_cols=["order_revenue_sgd", "shipping_revenue_sgd", "order_total_incl_shipping_sgd", "order_discount_sgd"],
)
validation_summary(
    "lines",
    lines,
    key_cols=["order_id", "line_id"],
    expected_grain="one row per Shopify product line item",
    required_cols=["order_id", "customer_id", "Line: Type", "Line: Title", "Line: Quantity", "Line: Total"],
    money_cols=["Line: Price", "Line: Discount", "Line: Total"],
)
validation_summary(
    "order_non_product_lines",
    non_product_lines,
    key_cols=["order_id", "Line: Type"],
    expected_grain="one row per non-product Shopify transaction/export row",
    required_cols=["order_id", "Line: Type"],
    money_cols=["Line: Price", "Line: Discount", "Line: Total"],
)
validation_summary(
    "products",
    products,
    key_cols=["product_variant_sku"],
    expected_grain="one row per product variant SKU",
    required_cols=["product_variant_sku", "product_handle", "product_category"],
    money_cols=["variant_price", "cost_per_item"],
)
validation_summary(
    "discounts",
    discounts,
    key_cols=["Name"],
    expected_grain="one row per discount code",
    required_cols=["Name", "Value", "Value Type", "Status"],
    money_cols=["Value", "Times Used In Total"],
)
validation_summary(
    "campaigns",
    campaigns,
    key_cols=["Referrer source", "Referrer name", "Session city", "UTM campaign", "UTM medium", "UTM source", "Landing page path", "Landing page URL"],
    expected_grain="one aggregate row per referrer/UTM/landing-page/city combination",
    required_cols=["Online store visitors", "Sessions"],
    money_cols=["Online store visitors", "Sessions"],
)
validation_summary(
    "recharge_orders",
    recharge_orders,
    key_cols=["recharge_order_id"],
    expected_grain="one row per Recharge order",
    required_cols=["recharge_order_id", "shopify_order_id", "customer_id"],
    money_cols=["order_total", "order_gross_revenue", "order_tax", "order_shipping", "order_discounts"],
)
validation_summary(
    "recharge_checkout_items",
    recharge_checkout_items,
    key_cols=["recharge_order_id", "product_sku"],
    expected_grain="one row per Recharge checkout order item; duplicate order/SKU pairs can occur when quantity or variants repeat",
    required_cols=["recharge_order_id", "shopify_order_id", "product_sku", "customer_id"],
    money_cols=["order_item_quantity", "line_item_price", "line_item_tax", "line_item_discount"],
)
validation_summary(
    "recharge_recurring_items",
    recharge_recurring_items,
    key_cols=["recharge_order_id", "product_sku"],
    expected_grain="one row per Recharge recurring order item; duplicate order/SKU pairs can occur when quantity or variants repeat",
    required_cols=["recharge_order_id", "shopify_order_id", "product_sku", "customer_id"],
    money_cols=["order_item_quantity", "line_item_price", "line_item_tax", "line_item_discount"],
)
validation_summary(
    "recharge_reactivated",
    recharge_reactivated,
    key_cols=["customer_id", "reactivated_date"],
    expected_grain="one row per subscriber reactivation event",
    required_cols=["customer_id", "reactivated_date"],
)
validation_summary(
    "recharge_churned",
    recharge_churned,
    key_cols=["subscription_id", "subscription_churn_date"],
    expected_grain="one row per churned subscription event",
    required_cols=["subscription_id", "customer_id", "subscription_churn_date", "cancellation_reason"],
)

assert orders["order_id"].is_unique, "silver/orders.parquet must be one row per order_id"
assert "line_items" in orders.columns, "orders should include array-based line-item summary"
assert orders["line_items"].map(lambda x: isinstance(x, (list, np.ndarray))).all(), "line_items must be stored as arrays/lists"
assert orders["line_item_skus"].map(lambda x: isinstance(x, (list, np.ndarray))).all(), "line_item_skus must be stored as arrays/lists"
assert np.isclose(orders["order_revenue_sgd"], orders["order_total_incl_shipping_sgd"] - orders["shipping_revenue_sgd"], rtol=0, atol=0.01).all(), "order revenue must equal total minus shipping"
assert lines["order_id"].isin(orders["order_id"]).mean() > 0.95, "most line items should map to cleaned orders"
assert len(non_product_lines) > 0, "non-product transaction rows should be preserved in Silver"
assert products["product_variant_sku"].is_unique, "products should be unique by product_variant_sku"
assert recharge_orders["recharge_order_id"].is_unique, "recharge_orders should be unique by recharge_order_id"
assert recharge_orders["shopify_order_id"].notna().all(), "recharge_orders should have shopify_order_id coverage"
assert not list(DATA_DIR.glob("*.parquet")), "Parquet files should live under data/silver or data/gold, not data root"
assert (SILVER_DIR / "README.md").exists(), "Silver README missing"

print("\nSilver validation passed.")
print("Orders include array-based line item summary and non-shipping revenue columns.")
print("\nAll Silver parquet files can be read back:")
for path in saved_paths:
    df = pd.read_parquet(path)
    print(f" - {path.relative_to(PROJECT_ROOT)}: {len(df):,} rows")
print("\nNon-product row types preserved for audit:")
print(non_product_lines["Line: Type"].value_counts(dropna=False).to_string())

Silver validation checks

[orders] 27,350 rows x 49 columns
 - expected grain: one row per cleaned Shopify order
 - required columns present: True
 - key columns: ['order_id']
 - rows with null key: 0 (0.0%)
 - duplicate key rows: 0
 - money column order_revenue_sgd: non-numeric values 0
 - money column shipping_revenue_sgd: non-numeric values 0
 - money column order_total_incl_shipping_sgd: non-numeric values 0
 - money column order_discount_sgd: non-numeric values 0

[lines] 50,963 rows x 42 columns
 - expected grain: one row per Shopify product line item
 - required columns present: True
 - key columns: ['order_id', 'line_id']
 - rows with null key: 0 (0.0%)
 - duplicate key rows: 0
 - money column Line: Price: non-numeric values 0
 - money column Line: Discount: non-numeric values 0
 - money column Line: Total: non-numeric values 0

[order_non_product_lines] 102,608 rows x 41 columns
 - expected grain: one row per non-product Shopify transaction/export row
 - required columns prese

# 10. Data Preview

In [27]:
orders_df = pd.read_parquet(SILVER_DIR / "orders.parquet")
print(orders_df.shape)
orders_df.head()

(27350, 49)


,order_id,Name,Tags,order_date,processed_at_sgt,store,customer_id,Currency,Price: Total Line Items,Price: Current Subtotal,Price: Subtotal,Price: Total Discount,Price: Total Shipping,Price: Current Total Shipping,Price: Current Total,Price: Total,Payment: Status,Order Fulfillment Status,Shipping: Country,Shipping: Country Code,Browser: UTM Source,Browser: UTM Medium,Browser: UTM Campaign,Browser: Referrer Domain,Cancelled At,Cancel: Reason,Line: Product Handle,Line: Title,Line: Variant Title,Line: SKU,Line: Price,Line: Quantity,source_file,source_row_number,original_currency,shipping_revenue_sgd,order_total_incl_shipping_sgd,order_revenue_sgd,order_discount_sgd,order_line_items_gross_sgd,channel,product_category,has_discount,is_subscription,is_first_order_tag,line_item_count,line_item_quantity,line_items,line_item_skus
0,4992746586367,LPSG-9541,NaN,2020-12-31 00:00:00+08:00,2020-12-31 03:27:50+08:00,SG,6423777902847,SGD,83.30,83.30,83.30,0.0,4.9,4.9,88.20,88.20,paid,fulfilled,Singapore,SG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Better Whey - 500g Pack, Chocolate Dinosaur",NaN,NaN,22.15,1.0,1_1.orders-2020_20260505.xlsx,2,SGD,4.9,88.20,83.30,0.0,83.30,Subscription,Unknown,False,False,False,4,4.0,"[1 x Better Whey - 500g Pack, Chocolate Dinosa...",[]
1,4992746619135,LPSG-9542,NaN,2020-12-29 00:00:00+08:00,2020-12-29 09:59:53+08:00,SG,6328286576895,SGD,78.20,78.20,78.20,0.0,4.9,4.9,83.10,83.10,paid,fulfilled,Singapore,SG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Better Whey - 1KG Pack, Vanilla",NaN,NaN,39.10,1.0,1_1.orders-2020_20260505.xlsx,11,SGD,4.9,83.10,78.20,0.0,78.20,Subscription,Unknown,False,False,False,2,2.0,"[1 x Better Whey - 1KG Pack, Vanilla, 1 x Bett...",[]
2,4992746684671,LPSG-9543,NaN,2020-12-29 00:00:00+08:00,2020-12-29 09:49:45+08:00,SG,6328286576895,SGD,66.48,66.48,66.48,0.0,4.9,4.9,71.38,71.38,paid,fulfilled,Singapore,SG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Better Whey - 1KG Pack, Teh Tarik",NaN,NaN,33.24,1.0,1_1.orders-2020_20260505.xlsx,16,SGD,4.9,71.38,66.48,0.0,66.48,Subscription,Unknown,False,False,False,2,2.0,"[1 x Better Whey - 1KG Pack, Teh Tarik, 1 x Be...",[]
3,4992746455295,LPSG-9540,NaN,2020-12-31 00:00:00+08:00,2020-12-31 21:35:26+08:00,SG,6424407998719,SGD,6854.40,6854.40,6854.40,0.0,0.0,0.0,6854.40,6854.40,paid,fulfilled,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Plant Protein - 480g Pack, Chocolate Dinosaur",NaN,NaN,285.60,12.0,1_1.orders-2020_20260505.xlsx,21,SGD,0.0,6854.40,6854.40,0.0,6854.40,Subscription,Unknown,False,False,False,2,24.0,"[12 x Plant Protein - 480g Pack, Chocolate Din...",[]
4,4992746979583,LPSG-9546,NaN,2020-12-29 00:00:00+08:00,2020-12-29 04:22:23+08:00,SG,6327668277503,SGD,745.20,745.20,745.20,0.0,0.0,0.0,745.20,745.20,paid,fulfilled,Singapore,SG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Better Whey - 1KG Pack, Natural (Unflavoured)",NaN,NaN,149.04,5.0,1_1.orders-2020_20260505.xlsx,25,SGD,0.0,745.20,745.20,0.0,745.20,Subscription,Unknown,False,False,False,1,5.0,"[5 x Better Whey - 1KG Pack, Natural (Unflavou...",[]


In [28]:
orders_non_product_lines_df = pd.read_parquet(SILVER_DIR / "order_non_product_lines.parquet")
print(orders_non_product_lines_df.shape)
orders_non_product_lines_df.head()

(102608, 41)


,order_id,customer_id,order_date,processed_at_sgt,store,Currency,Payment: Status,Order Fulfillment Status,Shipping: Zip,Shipping: City,Shipping: Province,Shipping: Country,Shipping: Country Code,line_id,Line: Type,Line: Product ID,Line: Product Handle,Line: Title,Line: Name,Line: Variant ID,Line: Variant Title,Line: SKU,Line: Quantity,Line: Price,Line: Discount,Line: Discount Allocation,Line: Discount per Item,Line: Total,Line: Grams,Line: Requires Shipping,Line: Vendor,Line: Gift Card,Line: Variant SKU,Line: Variant Barcode,Line: Variant Weight,Line: Variant Weight Unit,Line: Variant Inventory Qty,Line: Variant Cost,Line: Variant Price,source_file,source_row_number
0,4992746586367,6423777902847,2020-12-31 00:00:00+08:00,2020-12-31 03:27:50+08:00,SG,SGD,paid,fulfilled,461055,Singapore,NaN,Singapore,SG,4171771314431,Shipping Line,NaN,NaN,Tracked Shipping,Tracked Shipping,NaN,NaN,NaN,NaN,4.90,0.0,0.0,0.0,4.90,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1_1.orders-2020_20260505.xlsx,6
1,4992746586367,6423777902847,2020-12-31 00:00:00+08:00,2020-12-31 03:27:50+08:00,SG,SGD,paid,fulfilled,461055,Singapore,NaN,Singapore,SG,12664675926271,Fulfillment Line,NaN,NaN,"Better Whey - 500g Pack, Chocolate Dinosaur","Better Whey - 500g Pack, Chocolate Dinosaur",NaN,NaN,NaN,-1.0,22.15,0.0,0.0,0.0,-22.15,0.0,1.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1_1.orders-2020_20260505.xlsx,7
2,4992746586367,6423777902847,2020-12-31 00:00:00+08:00,2020-12-31 03:27:50+08:00,SG,SGD,paid,fulfilled,461055,Singapore,NaN,Singapore,SG,12664675959039,Fulfillment Line,NaN,NaN,Green Tea Extract Capsules - 60 Capsules,Green Tea Extract Capsules - 60 Capsules,NaN,NaN,NaN,-1.0,13.50,0.0,0.0,0.0,-13.50,0.0,1.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1_1.orders-2020_20260505.xlsx,8
3,4992746586367,6423777902847,2020-12-31 00:00:00+08:00,2020-12-31 03:27:50+08:00,SG,SGD,paid,fulfilled,461055,Singapore,NaN,Singapore,SG,12664675991807,Fulfillment Line,NaN,NaN,"Better Whey - 500g Pack, Mango","Better Whey - 500g Pack, Mango",NaN,NaN,NaN,-1.0,22.15,0.0,0.0,0.0,-22.15,0.0,1.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1_1.orders-2020_20260505.xlsx,9
4,4992746586367,6423777902847,2020-12-31 00:00:00+08:00,2020-12-31 03:27:50+08:00,SG,SGD,paid,fulfilled,461055,Singapore,NaN,Singapore,SG,12664676024575,Fulfillment Line,NaN,NaN,L-Carnitine,L-Carnitine,NaN,NaN,NaN,-1.0,25.50,0.0,0.0,0.0,-25.50,0.0,1.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1_1.orders-2020_20260505.xlsx,10


In [29]:
products_df = pd.read_parquet(SILVER_DIR / "products.parquet")
print(products_df.shape)
products_df.head()

(56, 25)


,product_handle,product_title,Body (HTML),vendor,product_variant_sku,Variant Grams,variant_price,Variant Compare At Price,Variant Barcode,Image Src,Gift Card,Variant Image,Variant Weight Unit,cost_per_item,status,Included / Singapore,Price / Singapore,Compare At Price / Singapore,Included / Hong Kong,Price / Hong Kong,Compare At Price / Hong Kong,Included / Malaysia,Price / Malaysia,Compare At Price / Malaysia,product_category
0,lean-protein-peach-oolong-pre-order,LEAN PROTEIN PEACH OOLONG (PRE-ORDER),<p>Elevate your fitness journey with our new L...,SG,LEAN-POL-1KG-V1,1000,59.9,<NA>,'0724999807825,https://cdn.shopify.com/s/files/1/0660/4895/05...,False,https://cdn.shopify.com/s/files/1/0660/4895/05...,kg,17.65,active,True,<NA>,<NA>,True,<NA>,<NA>,True,179.0,<NA>,Lean Protein
1,carton_20pc-clear-protein-25g-peach-pack-of-6,[CARTON_20PC] Clear Protein 25g Peach (Pack of 5),<NA>,lushprotein,CLEAR-PEA-25G-CTN20-5PK-V2,0,0.0,<NA>,<NA>,<NA>,False,<NA>,kg,83.0,active,True,<NA>,<NA>,True,<NA>,<NA>,True,<NA>,<NA>,Clear Protein
2,carton_20pc-clear-protein-25g-peach-pack-of-5,[CARTON_20PC] Clear Protein 25g Grape (Pack of 5),<p><span>724999809641</span></p>,lushprotein,CLEAR-GRA-25G-CTN20-5PK-V2,0,0.0,<NA>,<NA>,<NA>,False,<NA>,kg,83.0,active,True,<NA>,<NA>,True,<NA>,<NA>,True,<NA>,<NA>,Clear Protein
3,bag_20pc-clear-protein-peach-40g,[BAG_20PC] Clear Protein Peach 25g,<NA>,lushprotein,CLEAR-PEA-25G-20PK-V2,0,0.0,<NA>,<NA>,<NA>,False,<NA>,kg,21.0,active,True,<NA>,<NA>,True,<NA>,<NA>,True,<NA>,<NA>,Clear Protein
4,bag_20pc-clear-protein-grape-40g,[BAG_20PC] Clear Protein Grape 25g,<NA>,lushprotein,CLEAR-GRA-25G-20PK-V2,0,0.0,<NA>,<NA>,<NA>,False,<NA>,kg,21.0,active,True,<NA>,<NA>,True,<NA>,<NA>,True,<NA>,<NA>,Clear Protein


In [30]:
lines_df = pd.read_parquet(SILVER_DIR / "lines.parquet")
print(lines_df.shape)
lines_df.head()

(50963, 42)


,order_id,customer_id,order_date,processed_at_sgt,store,Currency,Payment: Status,Order Fulfillment Status,Shipping: Zip,Shipping: City,Shipping: Province,Shipping: Country,Shipping: Country Code,line_id,Line: Type,Line: Product ID,Line: Product Handle,Line: Title,Line: Name,Line: Variant ID,Line: Variant Title,Line: SKU,Line: Quantity,Line: Price,Line: Discount,Line: Discount Allocation,Line: Discount per Item,Line: Total,Line: Grams,Line: Requires Shipping,Line: Vendor,Line: Gift Card,Line: Variant SKU,Line: Variant Barcode,Line: Variant Weight,Line: Variant Weight Unit,Line: Variant Inventory Qty,Line: Variant Cost,Line: Variant Price,source_file,source_row_number,product_category
0,4992746586367,6423777902847,2020-12-31 00:00:00+08:00,2020-12-31 03:27:50+08:00,SG,SGD,paid,fulfilled,461055,Singapore,NaN,Singapore,SG,12664675926271,Line Item,NaN,NaN,"Better Whey - 500g Pack, Chocolate Dinosaur","Better Whey - 500g Pack, Chocolate Dinosaur",NaN,NaN,NaN,1.0,22.15,0.0,0.0,0.0,22.15,0.0,1.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1_1.orders-2020_20260505.xlsx,2,Unknown
1,4992746586367,6423777902847,2020-12-31 00:00:00+08:00,2020-12-31 03:27:50+08:00,SG,SGD,paid,fulfilled,461055,Singapore,NaN,Singapore,SG,12664675959039,Line Item,NaN,NaN,Green Tea Extract Capsules - 60 Capsules,Green Tea Extract Capsules - 60 Capsules,NaN,NaN,NaN,1.0,13.50,0.0,0.0,0.0,13.50,0.0,1.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1_1.orders-2020_20260505.xlsx,3,Unknown
2,4992746586367,6423777902847,2020-12-31 00:00:00+08:00,2020-12-31 03:27:50+08:00,SG,SGD,paid,fulfilled,461055,Singapore,NaN,Singapore,SG,12664675991807,Line Item,NaN,NaN,"Better Whey - 500g Pack, Mango","Better Whey - 500g Pack, Mango",NaN,NaN,NaN,1.0,22.15,0.0,0.0,0.0,22.15,0.0,1.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1_1.orders-2020_20260505.xlsx,4,Unknown
3,4992746586367,6423777902847,2020-12-31 00:00:00+08:00,2020-12-31 03:27:50+08:00,SG,SGD,paid,fulfilled,461055,Singapore,NaN,Singapore,SG,12664676024575,Line Item,NaN,NaN,L-Carnitine,L-Carnitine,NaN,NaN,NaN,1.0,25.50,0.0,0.0,0.0,25.50,0.0,1.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1_1.orders-2020_20260505.xlsx,5,Unknown
4,4992746619135,6328286576895,2020-12-29 00:00:00+08:00,2020-12-29 09:59:53+08:00,SG,SGD,paid,fulfilled,328358,Singapore,NaN,Singapore,SG,12664676122879,Line Item,NaN,NaN,"Better Whey - 1KG Pack, Vanilla","Better Whey - 1KG Pack, Vanilla",NaN,NaN,NaN,1.0,39.10,0.0,0.0,0.0,39.10,0.0,1.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1_1.orders-2020_20260505.xlsx,11,Unknown


In [31]:
discounts_df = pd.read_parquet(SILVER_DIR / "discounts.parquet")
print(discounts_df.shape)
discounts_df.head()

(367, 17)


,Name,Value,Value Type,Type,Discount Class,Minimum Purchase Requirements,Combines with Order Discounts,Combines with Product Discounts,Combines with Shipping Discounts,Customer Selection,Context,Times Used In Total,Applies Once Per Customer,Usage Limit Per Code,Status,Start,End
0,SG40,-40.0,percentage,Amount Off,order,NaN,NaN,NaN,NaN,all,all,5,NaN,NaN,Expired,2022-09-18 06:39:30+00:00,2022-10-13 02:08:02+00:00
1,september,-58.0,percentage,Amount Off,order,NaN,NaN,NaN,NaN,all,all,72,NaN,NaN,Expired,2022-09-24 01:30:00+00:00,2022-10-01 04:00:00+00:00
2,9f9abe1f833f,-20.0,percentage,Amount Off,order,10.0,NaN,NaN,NaN,all,all,0,NaN,1.0,Active,2022-09-23 18:06:29+00:00,NaT
3,bftksept,-20.0,percentage,Amount Off,order,NaN,NaN,NaN,NaN,all,all,0,NaN,NaN,Expired,2022-09-26 05:29:26+00:00,2022-09-30 16:00:29+00:00
4,ACF,-20.0,percentage,Amount Off,order,NaN,NaN,NaN,NaN,all,all,0,NaN,NaN,Expired,2022-09-26 10:55:50+00:00,2022-12-31 15:59:55+00:00


In [32]:
campaigns_df = pd.read_parquet(SILVER_DIR / "campaigns.parquet")
print(campaigns_df.shape)
campaigns_df.head()

(137033, 10)


,Referrer source,Referrer name,Session city,UTM campaign,UTM medium,UTM source,Landing page path,Landing page URL,Online store visitors,Sessions
0,search,google,Singapore,NaN,NaN,NaN,/,https://lushprotein.com/,3680,4408
1,direct,NaN,NaN,NaN,NaN,NaN,/,https://www.lushprotein.com/,3340,4308
2,direct,NaN,Singapore,NaN,NaN,NaN,/,https://lushprotein.com/,2713,3972
3,search,google,Singapore,NaN,NaN,NaN,/,https://www.lushprotein.com/,2538,3043
4,direct,NaN,Singapore,NaN,NaN,NaN,/,https://www.lushprotein.com/,1918,2931


In [33]:
recharge_orders_df = pd.read_parquet(SILVER_DIR / "recharge_orders.parquet")
print(recharge_orders_df.shape)
recharge_orders_df.head()

(1215, 10)


,metric_date,recharge_order_id,shopify_order_id,order_type,order_total,order_gross_revenue,order_tax,order_shipping,order_discounts,customer_id
0,2026-04-07,1306407778,6789789155583,recurring,62.1,62.1,0,0.0,0.0,237412247
1,2026-04-07,1306407662,6789789057279,recurring,107.82,107.82,0,0.0,0.0,237898931
2,2026-04-07,1306910933,6790149636351,checkout,128.01,142.23,0,0.0,-14.22,243992454
3,2026-04-07,1307273448,6791342620927,checkout,126.08,141.14,0,0.0,-15.06,244069323
4,2026-04-07,1306407554,6789789024511,recurring,33.07,33.07,0,0.0,0.0,175122062


In [34]:
recharge_checkout_items_df = pd.read_parquet(SILVER_DIR / "recharge_checkout_items.parquet")
print(recharge_checkout_items_df.shape)
recharge_checkout_items_df.head()

(1094, 14)


,metric_date,recharge_order_id,shopify_order_id,product_id,variant_id,product_sku,purchase_type,product_title,variant_title,order_item_quantity,line_item_price,line_item_tax,line_item_discount,customer_id
0,2026-04-07,1307273448,6791342620927,7792625680639,46620967305471,ACC-SHK-V2,onetime,LP CLASSIC SHAKER,White,1,9.46,0,-1.89,244069323
1,2026-04-07,1306657882,6789900239103,8326528991487,44602471940351,CLEAR-GRA-500G-V2,subscription,CLEAR PROTEIN,1 x 500g Pack / White Grape,1,66.24,0,0.0,243978193
2,2026-04-07,1307273448,6791342620927,8326528991487,44602471940351,CLEAR-GRA-500G-V2,subscription,CLEAR PROTEIN,1 x 500g Pack / White Grape,2,131.68,0,-13.17,244069323
3,2026-04-07,1306657882,6789900239103,7792625287423,46620966781183,CRE-UNF-250G-V1,subscription,CREATINE MONOHYDRATE,1 x 250g Pack (50 servings),1,20.87,0,-2.09,243978193
4,2026-04-07,1306657882,6789900239103,8548958470399,45301479702783,LEAN-THA-1KG-V1,subscription,LEAN PROTEIN,1 x 1kg Pack / Thai Milk Tea,1,54.14,0,0.0,243978193


In [35]:
recharge_churned_df = pd.read_parquet(SILVER_DIR / "recharge_churned.parquet")
print(recharge_churned_df.shape)
recharge_churned_df.head()

(526, 12)


,metric_date,subscription_id,customer_id,product_id,product_title,variant_id,product_sku,variant_title,subscription_activation_date,subscription_churn_date,churn_type,cancellation_reason
0,2026-04-07,767150230,233442228,8326528991487,CLEAR PROTEIN,44602471907583,CLEAR-PEA-500G-V2,1 x 500g Pack / Peach,2026-02-17,2026-04-07,active,This was created by accident
1,2026-04-07,580604275,175122113,8548958470399,LEAN PROTEIN,45301479702783,LEAN-THA-1KG-V1,1 x 1kg Pack / Thai Milk Tea,2025-01-16,2026-04-07,active,I no longer use this product
2,2026-04-07,777767913,237898931,8548958470399,LEAN PROTEIN,46371756998911,LEAN-TAR-1KG-V1,1 x 1kg Pack / Taro,2026-03-10,2026-04-07,active,This is too expensive
3,2026-04-07,767150231,233442228,8548958470399,LEAN PROTEIN,45301479702783,LEAN-THA-1KG-V1,1 x 1kg Pack / Thai Milk Tea,2026-02-17,2026-04-07,active,This was created by accident
4,2026-04-06,762373289,231320604,8326528991487,CLEAR PROTEIN,44602471907583,CLEAR-PEA-500G-V2,1 x 500g Pack / Peach,2026-02-06,2026-04-06,active,(Unknown)


In [36]:
recharge_recurring_items_df = pd.read_parquet(SILVER_DIR / "recharge_recurring_items.parquet")
print(recharge_recurring_items_df.shape)
recharge_recurring_items_df.head()

(650, 14)


,metric_date,recharge_order_id,shopify_order_id,product_id,variant_id,product_sku,purchase_type,product_title,variant_title,order_item_quantity,line_item_price,line_item_tax,line_item_discount,customer_id
0,2026-04-07,1306407490,6789788991743,8548958470399,46371756998911,LEAN-TAR-1KG-V1,subscription,LEAN PROTEIN,1 x 1kg Pack / Taro,1,56.91,0,0.0,231695435
1,2026-04-07,1306407605,6789789090047,8326528991487,44602471907583,CLEAR-PEA-500G-V2,subscription,CLEAR PROTEIN,1 x 500g Pack / Peach,2,113.02,0,0.0,237873185
2,2026-04-07,1306407554,6789789024511,7792627024127,46752160481535,CAP-MUL-BOTTLE-V1,subscription,MULTIVITAMIN CAPSULES,<NA>,1,33.07,0,0.0,175122062
3,2026-04-07,1306407490,6789788991743,8326528991487,44602471940351,CLEAR-GRA-500G-V2,subscription,CLEAR PROTEIN,1 x 500g Pack / White Grape,1,65.55,0,0.0,231695435
4,2026-04-07,1306407662,6789789057279,8548958470399,46371756998911,LEAN-TAR-1KG-V1,subscription,LEAN PROTEIN,1 x 1kg Pack / Taro,2,107.82,0,0.0,237898931


In [37]:
recharge_reactivated_df = pd.read_parquet(SILVER_DIR / "recharge_reactivated.parquet")
print(recharge_reactivated_df.shape)
recharge_reactivated_df.head()

(50, 4)


,metric_date,customer_id,first_subscription_activation_date,reactivated_date
0,2026-04-05,175122042,2025-01-16,2026-04-05
1,2026-04-01,191230863,2025-04-09,2026-04-01
2,2026-04-01,230786752,2026-02-02,2026-04-01
3,2026-03-27,214043366,2025-10-10,2026-03-27
4,2026-03-22,204781914,2025-07-22,2026-03-22
